# Segmentação de Folhas com Segment Anything Model (SAM)

Este notebook contém a comparação e testes práticos de segmentação de folhas utilizando duas versões do **Segment Anything Model (SAM)** da Meta:
1. **SAM 1 (Base)** (`facebook/sam-vit-base`): O modelo original base, altamente otimizado para CPU.
2. **SAM 2.1 (Tiny)** (`facebook/sam2.1-hiera-tiny`): A versão mais recente e leve baseada no Hiera ViT, porém ainda não otimizada para CPU na biblioteca `transformers`.

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
import torch
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import time
from transformers import pipeline

# Carrega variáveis do .env (como HF_TOKEN)
load_dotenv(find_dotenv())

print("Dispositivo para execução:", "CUDA/GPU" if torch.cuda.is_available() else "CPU")

In [ ]:
# Carrega a imagem da folha e redimensiona para otimizar velocidade no CPU
image_path = "../folha1.jpg"
image = Image.open(image_path)

# Redimensiona mantendo a proporção (máximo 512px na maior dimensão para testes rápidos no CPU)
max_size = 512
ratio = min(max_size / image.width, max_size / image.height)
if ratio < 1.0:
    new_size = (int(image.width * ratio), int(image.height * ratio))
    image_resized = image.resize(new_size, Image.Resampling.LANCZOS)
    print(f"Imagem carregada e redimensionada de {image.size} para {image_resized.size}")
else:
    image_resized = image

plt.figure(figsize=(6, 6))
plt.imshow(image_resized)
plt.title("Imagem Original (Redimensionada)")
plt.axis("off")
plt.show()

## 1. Teste de Segmentação com SAM 1 (Base)

O modelo `facebook/sam-vit-base` tem um tamanho de cerca de 375 MB. Ele possui uma excelente otimização no PyTorch para execuções em CPU.

In [ ]:
print("Carregando o SAM 1 (Base)...")
t0 = time.time()
generator_sam1 = pipeline("mask-generation", model="facebook/sam-vit-base", device=-1)
print(f"Modelo carregado em {time.time() - t0:.2f}s")

print("Gerando máscaras...")
t1 = time.time()
outputs_sam1 = generator_sam1(image_resized)
masks_sam1 = outputs_sam1["masks"]
print(f"Total de {len(masks_sam1)} máscaras geradas em {time.time() - t1:.2f}s!")

In [ ]:
# Ordena as máscaras por área decrescente
masks_sorted_sam1 = sorted(masks_sam1, key=lambda m: np.array(m).sum(), reverse=True)

# Seleciona a maior máscara que não seja o fundo completo
best_mask_sam1 = None
for m in masks_sorted_sam1:
    mask_np = np.array(m) > 0
    if mask_np.sum() < (0.95 * mask_np.size):
        best_mask_sam1 = mask_np
        break

if best_mask_sam1 is not None:
    img_np = np.array(image_resized)
    
    # Aplica a máscara (fundo preto)
    segmented_img = img_np.copy()
    segmented_img[~best_mask_sam1] = 0
    
    # Recorta para a bounding box da folha
    rows, cols = np.where(best_mask_sam1)
    ymin, ymax = rows.min(), rows.max()
    xmin, xmax = cols.min(), cols.max()
    cropped_img = segmented_img[ymin:ymax+1, xmin:xmax+1]
    
    # Visualização
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(image_resized)
    axes[0].set_title("Original")
    axes[0].axis("off")
    
    axes[1].imshow(best_mask_sam1, cmap="gray")
    axes[1].set_title("Máscara (SAM 1)")
    axes[1].axis("off")
    
    axes[2].imshow(cropped_img)
    axes[2].set_title("Recorte da Folha")
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()
else:
    print("Nenhuma máscara adequada encontrada.")

## 2. Teste de Segmentação com SAM 2.1 (Tiny)

O modelo `facebook/sam2.1-hiera-tiny` possui pesos de apenas 156 MB, porém seu processo de inferência em CPU costuma ser mais lento devido à falta de otimizações de kernels na biblioteca `transformers`.

In [ ]:
print("Carregando o SAM 2.1 (Tiny)...")
t0 = time.time()
generator_sam2 = pipeline("mask-generation", model="facebook/sam2.1-hiera-tiny", device=-1)
print(f"Modelo carregado em {time.time() - t0:.2f}s")

print("Gerando máscaras...")
t1 = time.time()
outputs_sam2 = generator_sam2(image_resized)
masks_sam2 = outputs_sam2["masks"]
print(f"Total de {len(masks_sam2)} máscaras geradas em {time.time() - t1:.2f}s!")

In [ ]:
# Ordena as máscaras por área decrescente
masks_sorted_sam2 = sorted(masks_sam2, key=lambda m: np.array(m).sum(), reverse=True)

# Seleciona a maior máscara que não seja o fundo completo
best_mask_sam2 = None
for m in masks_sorted_sam2:
    mask_np = np.array(m) > 0
    if mask_np.sum() < (0.95 * mask_np.size):
        best_mask_sam2 = mask_np
        break

if best_mask_sam2 is not None:
    img_np = np.array(image_resized)
    
    # Aplica a máscara (fundo preto)
    segmented_img = img_np.copy()
    segmented_img[~best_mask_sam2] = 0
    
    # Recorta para a bounding box da folha
    rows, cols = np.where(best_mask_sam2)
    ymin, ymax = rows.min(), rows.max()
    xmin, xmax = cols.min(), cols.max()
    cropped_img = segmented_img[ymin:ymax+1, xmin:xmax+1]
    
    # Visualização
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(image_resized)
    axes[0].set_title("Original")
    axes[0].axis("off")
    
    axes[1].imshow(best_mask_sam2, cmap="gray")
    axes[1].set_title("Máscara (SAM 2.1)")
    axes[1].axis("off")
    
    axes[2].imshow(cropped_img)
    axes[2].set_title("Recorte da Folha")
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()
else:
    print("Nenhuma máscara adequada encontrada.")